# 04 — Sparse-GAT-ViT Training
Main contribution: sparse k-NN graph with Gaussian edge weights + pruning threshold.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))


In [ ]:
import torch
from src.models import SparseGATViT

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SparseGATViT(num_classes=100, graph_k=4, threshold=0.1).to(device)
n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Sparse-GAT-ViT params: {n/1e6:.2f}M')

dummy = torch.randn(2, 3, 128, 128).to(device)
with torch.no_grad():
    out = model(dummy)
    avg_edges = model.edge_count_per_sample(dummy)
print(f'Output: {out.shape}  |  Avg edges/graph: {avg_edges:.1f}')

In [ ]:
# Edge count vs k ablation (sparse vs dense)
import matplotlib.pyplot as plt
import numpy as np

k_values = [1, 2, 3, 4, 6, 8]
edge_counts = []

dummy_single = torch.randn(1, 3, 128, 128).to(device)
for k in k_values:
    m = SparseGATViT(num_classes=100, graph_k=k, threshold=0.1).to(device)
    with torch.no_grad():
        ec = m.edge_count_per_sample(dummy_single)
    edge_counts.append(ec)
    print(f'k={k}: avg edges = {ec:.1f}')

# Dense 8-neighbor reference
nH = nW = 128 // 4 // 4  # depends on backbone downsampling
# just plot the sparse values
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, edge_counts, 'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('k (nearest neighbours)')
ax.set_ylabel('Average edges per graph')
ax.set_title('Edge Count vs k in Sparse Graph Construction')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/edge_count_vs_k.png', dpi=120)
plt.show()

In [ ]:
# Train (run from CLI for full 100 epochs)
# !python ../src/train.py --model sparse_gat --epochs 100 --batch 64 --out ../results --data ../data
from pathlib import Path
log_path = '../results/sparse_gat/train_log.csv'
if Path(log_path).exists():
    from src.visualize import plot_training_curves
    plot_training_curves(log_path, Path('../results/sparse_gat'))
    from IPython.display import Image
    display(Image('../results/sparse_gat/training_curves.png'))
else:
    print('Run training first.')